In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from openai import OpenAI


In [2]:
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [3]:
df_pos = pd.read_csv("tashbeeh_dataset.csv")
df_neg = pd.read_csv("non_tashbeeh_random_sample.csv")

df_pos["label"] = 1  # tashbeeh
df_neg["label"] = 0  # non-tashbeeh

TEXT_COL = "البيت_الكامل"

df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df[[TEXT_COL, "label"]].dropna()

df.head()


,البيت_الكامل,label
0,أهُمُّ بِشَيْءٍ واللَّيالي كأنَّهَا تُطارِدُنِ...,1
1,تَثَنَّى عَلى قَدْرِ الطِّعانِ كأنَّما مَفاصِل...,1
2,وَلمَّا رأيْتُ النَّاسَ دُونَ مَحَلِّهِ تَيَقّ...,1
3,مُخَضَّبَةٌ والْقَوْمُ صَرْعَى كأنَّها وَإنْ ل...,1
4,تُنَكِّسِهُمْ والسَّابِقاتُ جِبالُهُمْ وَتَطْع...,1


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label"].values,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

len(X_train), len(X_test)


(872, 218)

In [5]:
K = 8  # total examples in prompt (balanced)

train_df = pd.DataFrame({
    "verse": X_train,
    "label": y_train
})

pos_examples = train_df[train_df.label == 1].sample(K // 2, random_state=42)
neg_examples = train_df[train_df.label == 0].sample(K // 2, random_state=42)

few_shot_examples = pd.concat([pos_examples, neg_examples]).sample(frac=1, random_state=42)
few_shot_examples


,verse,label
149,كَثُرَتْ بهِ حَرَكاتُها ولقَدْ تُرَى مِنْ فَتْ...,1
164,لقدْ كُنْتُ أرجو أنْ تَجودي بنائلٍ فَأَخْلَفَ ...,0
783,ما دُمْتَ مِنْ أرَبِ الحِسانِ فإنَّما رَوْقُ ا...,1
498,بِأحْسَنَ مِنْها مُقْلَةً ومُقَلَّدًا إذا جُلِ...,0
553,كافأْنَنا عنْ شِبْههنَّ مِنَ المَها فَلَهُنَّ ...,1
565,تَرَحّلَ مِنْ أرْضِ العرَاقِ مُرَقِّشٌ على طَر...,0
598,لِلَّهْوِ آوِنَةٌ تَمُرُّ كَأنَّها قُبَلٌ يُزَ...,1
225,يا رُبَّ ناكث بيعتين تَركته وخضاب لحيته دم الأ...,0


In [6]:
def build_few_shot_prompt(test_verse, examples_df):
    prompt = """
You are an expert in Arabic rhetoric (البلاغة العربية).

Task:
Determine whether a verse contains tashbeeh (تشبيه).

Rules:
- Output ONLY one number
- 1 = contains tashbeeh
- 0 = does NOT contain tashbeeh

Examples:
"""

    for _, row in examples_df.iterrows():
        prompt += f"""
Verse:
{row.verse}
Label: {row.label}
"""

    prompt += f"""
Now classify the following verse:

Verse:
{test_verse}
Label:
"""

    return prompt


In [7]:
def predict_tashbeeh_fewshot(test_verse, examples_df):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": build_few_shot_prompt(test_verse, examples_df)
            }
        ],
        temperature=0
    )

    output = response.choices[0].message.content.strip()
    return 1 if output == "1" else 0


In [8]:
y_pred_fewshot = []

for verse in tqdm(X_test):
    try:
        y_pred_fewshot.append(
            predict_tashbeeh_fewshot(verse, few_shot_examples)
        )
    except Exception as e:
        print("API error:", e)
        y_pred_fewshot.append(0)


100%|██████████| 218/218 [02:16<00:00,  1.59it/s]


In [9]:
print("Few-shot Accuracy:", accuracy_score(y_test, y_pred_fewshot))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_fewshot))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_fewshot,
    target_names=["Non-Tashbeeh", "Tashbeeh"]
))


Few-shot Accuracy: 0.7522935779816514

Confusion Matrix:
[[83 26]
 [28 81]]

Classification Report:
              precision    recall  f1-score   support

Non-Tashbeeh       0.75      0.76      0.75       109
    Tashbeeh       0.76      0.74      0.75       109

    accuracy                           0.75       218
   macro avg       0.75      0.75      0.75       218
weighted avg       0.75      0.75      0.75       218



In [10]:
results_df = pd.DataFrame({
    "verse": X_test,
    "y_true": y_test,
    "y_pred": y_pred_fewshot
})

results_df["model"] = "gpt-4o-mini"
results_df["prompt_type"] = "few-shot"
results_df["num_examples"] = K
results_df["temperature"] = 0

results_df.to_csv(
    "gpt_fewshot_tashbeeh_test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved results to gpt_fewshot_tashbeeh_test.csv")
results_df.head()


Saved results to gpt_fewshot_tashbeeh_test.csv


,verse,y_true,y_pred,model,prompt_type,num_examples,temperature
0,تَسَلَّت عمَايَاتُ الرِّجالِ عن الصِّبَا وَلَي...,0,0,gpt-4o-mini,few-shot,8,0
1,وَلذا اسْمُ أغْطِيةِ العُيُونِ جُفونها مِنْ أن...,1,1,gpt-4o-mini,few-shot,8,0
2,يَغْدُو مُؤَمِّلُهُ إِذَا ما حَطَّ في أَكْنَاف...,1,0,gpt-4o-mini,few-shot,8,0
3,بمُخضَّبٍ رخصٍ، كأنّ بنانهُ عَنمٌ، يكادُ منَ ا...,1,1,gpt-4o-mini,few-shot,8,0
4,كَمْ نِيلَ تحتَ سَناهَا مِن سَنا قمَرٍ وتَحْتَ...,0,1,gpt-4o-mini,few-shot,8,0
